# 2. Comparación emparejada de calidad
Ejecutar tras el notebook 01. Solo lee las trazas de test exportadas por los adaptadores y usa el evaluador común. Si difieren timestamps, etiquetas o manifiesto, falla antes de mostrar métricas.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import sys
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT
from shared.evaluation.evaluate import evaluate_pair
assert Path.cwd().resolve() == ROOT
traces = ROOT / 'results' / 'traces'
report = evaluate_pair(traces / 'lightgbm_night_test.parquet', traces / 'lightgbm_night_model2_isolated_test.parquet', traces / 'lstm_test.parquet', traces / 'lstm_model2_isolated_test.parquet')
assert report['paired_test_rows'] == 31680
print('Objetivos emparejados:', report['paired_test_rows'])


Objetivos emparejados: 31680


In [2]:
sections = [('Model 1 binario', 'binary_full_test'), ('Model 2 aislado', 'model2_isolated_true_anomalies'), ('Model 2 condicionado', 'model2_conditional_detected_true_anomalies'), ('Cascada 7 clases', 'cascade_seven_class_full_test')]
rows = []
for approach in ('lightgbm', 'lstm'):
    for label, key in sections:
        item = report[approach][key]
        rows.append({'enfoque': approach, 'evaluación': label, 'objetivos': item['rows'], 'macro_precision': item['per_class']['macro avg']['precision'], 'macro_recall': item['per_class']['macro avg']['recall'], 'macro_f1': item['per_class']['macro avg']['f1-score']})
display(pd.DataFrame(rows))
display(pd.DataFrame([{'enfoque': a, 'falsas_alarmas_model1': report[a]['false_alarms_model1'], 'anomalias_omitidas_model1': report[a]['missed_anomalies_model1']} for a in ('lightgbm', 'lstm')]))


,enfoque,evaluación,objetivos,macro_precision,macro_recall,macro_f1
0,lightgbm,Model 1 binario,31680,0.934770,0.923920,0.929244
1,lightgbm,Model 2 aislado,4395,0.909430,0.848592,0.844782
2,lightgbm,Model 2 condicionado,3801,0.948191,0.868442,0.881898
3,lightgbm,Cascada 7 clases,31680,0.880896,0.817243,0.815229
4,lstm,Model 1 binario,31680,0.966249,0.935999,0.950387
5,lstm,Model 2 aislado,4395,0.851774,0.815992,0.788542
6,lstm,Model 2 condicionado,3864,0.884151,0.846606,0.857582
7,lstm,Cascada 7 clases,31680,0.858092,0.829158,0.820618


,enfoque,falsas_alarmas_model1,anomalias_omitidas_model1
0,lightgbm,464,594
1,lstm,196,531


In [3]:
for approach in ('lightgbm', 'lstm'):
    print('\n', approach, report[approach]['model_version'])
    for label, key in sections:
        item = report[approach][key]
        print(label)
        display(pd.DataFrame(item['confusion_matrix'], index=item['labels'], columns=item['labels']))
        display(pd.DataFrame(item['per_class']).T.loc[list(item['labels']), ['precision', 'recall', 'f1-score', 'support']])



 lightgbm lightgbm-night-context-v1-20261104T114333Z
Model 1 binario


,Normal,Anomalous
Normal,26821,464
Anomalous,594,3801


,precision,recall,f1-score,support
Normal,0.978333,0.982994,0.980658,27285.0
Anomalous,0.891208,0.864846,0.877829,4395.0


Model 2 aislado


,Contextual,Noise,Correlation deviation,Missing data,Out-of-range,Stuck sensor
Contextual,704,0,1,0,0,0
Noise,2,878,62,0,0,8
Correlation deviation,15,53,876,0,0,6
Missing data,0,0,0,633,0,0
Out-of-range,0,0,0,0,634,0
Stuck sensor,0,354,40,0,0,129


,precision,recall,f1-score,support
Contextual,0.976422,0.998582,0.987377,705.0
Noise,0.683268,0.924211,0.785682,950.0
Correlation deviation,0.894791,0.922105,0.908243,950.0
Missing data,1.000000,1.000000,1.000000,633.0
Out-of-range,1.000000,1.000000,1.000000,634.0
Stuck sensor,0.902098,0.246654,0.387387,523.0


... (14 outputs left out)

,precision,recall,f1-score,support
Contextual,0.988717,0.994326,0.991513,705.0
Noise,0.947557,0.879425,0.912220,904.0
Correlation deviation,0.890304,0.975322,0.930876,932.0
Missing data,0.998420,0.998420,0.998420,633.0
Out-of-range,0.998425,1.000000,0.999212,634.0
Stuck sensor,0.481481,0.232143,0.313253,56.0


Cascada 7 clases


,Normal,Contextual,Noise,Correlation deviation,Missing data,Out-of-range,Stuck sensor
Normal,27089,9,115,65,0,0,7
Contextual,0,701,2,2,0,0,0
Noise,46,3,795,94,1,0,11
Correlation deviation,18,2,18,909,0,0,3
Missing data,0,0,0,0,632,1,0
Out-of-range,0,0,0,0,0,634,0
Stuck sensor,467,3,24,16,0,0,13


,precision,recall,f1-score,support
Normal,0.980775,0.992817,0.986759,27285.0
Contextual,0.976323,0.994326,0.985242,705.0
Noise,0.833333,0.836842,0.835084,950.0
Correlation deviation,0.837017,0.956842,0.892927,950.0
Missing data,0.998420,0.998420,0.998420,633.0
Out-of-range,0.998425,1.000000,0.999212,634.0
Stuck sensor,0.382353,0.024857,0.046679,523.0


El informe completo queda en `results/metrics/paired_test.json`, ignorado por Git. No mezclar Model 2 aislado, condicionado y la cascada: usan poblaciones diferentes.
